In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib
import os

from sklearn import set_config
from sklearn.model_selection import train_test_split

from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv

# Gradient Boosting

### Gradient Descent vs Boosting
- gradient descent: update parameters in the direction of the gradient
- gradient boosting: add a new model that points in the direction of the gradient

### Gradient Boosting vs Random Survival Forests
- RSF: survival trees + bagging + averaging survival functions
- CoxBoost: regression trees + boosting + Cox partial likelihood loss

### Base Learners
- regression trees

### Loss Function
- Cox proportional hazard function

## Preparing Data for Fitting

In [3]:
# load pre-processed datasets
train_data = pd.read_csv("../datasets/csv_files/ml_extension/train_data.csv")
test_data_one = pd.read_csv("../datasets/csv_files/ml_extension/test_data_one.csv")
test_data_two = pd.read_csv("../datasets/csv_files/ml_extension/test_data_two.csv")
test_data_three = pd.read_csv("../datasets/csv_files/ml_extension/test_data_three.csv")

In [4]:
# define X & y for all datasets
X_train = train_data[list(train_data.columns[3:])]
y_train = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', train_data)

X_test_one = test_data_one[list(test_data_one.columns[3:])]
X_test_one = X_test_one[X_train.columns]
y_test_one = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', test_data_one)

X_test_two = test_data_two[list(test_data_two.columns[3:])]
X_test_two = X_test_two[X_train.columns]
y_test_two = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', test_data_two)

X_test_three = test_data_three[list(test_data_three.columns[3:])]
X_test_three = X_test_three[X_train.columns]
y_test_three = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', test_data_three)

## Find Best n_estimators

In [5]:
class EarlyStoppingMonitor:
    def __init__(self, window_size, max_iter_without_improvement):
        self.window_size = window_size
        self.max_iter_without_improvement = max_iter_without_improvement
        self._best_step = -1

    def __call__(self, iteration, estimator, args):
        # continue training for first self.window_size iterations
        if iteration < self.window_size:
            return False

        # compute average improvement in last self.window_size iterations.
        # oob_improvement_ is the different in negative log partial likelihood
        # between the previous and current iteration.
        start = iteration - self.window_size + 1
        end = iteration + 1
        improvement = np.mean(estimator.oob_improvement_[start:end])

        if improvement > 1e-6:
            self._best_step = iteration
            return False  # continue fitting

        # stop fitting if there was no improvement
        # in last max_iter_without_improvement iterations
        diff = iteration - self._best_step
        return diff >= self.max_iter_without_improvement


est_early_stopping = GradientBoostingSurvivalAnalysis(
    n_estimators=1000, learning_rate=0.1, subsample=0.5, max_depth=1, random_state=0
)

monitor = EarlyStoppingMonitor(10, 5)

est_early_stopping.fit(X_train, y_train, monitor=monitor)

print("Fitted base learners:", est_early_stopping.n_estimators_)

cindex_one = est_early_stopping.score(X_test_one, y_test_one)
cindex_two = est_early_stopping.score(X_test_two, y_test_two)
cindex_three = est_early_stopping.score(X_test_three, y_test_three)
print("Performance on test set 1:", round(cindex_one, 3))
print("Performance on test set 2:", round(cindex_two, 3))
print("Performance on test set 3:", round(cindex_three, 3))

Fitted base learners: 62
Performance on test set 1: 0.623
Performance on test set 2: 0.563
Performance on test set 3: 0.481


In [6]:
# x, y = zip(*scores_cph_tree.items())
# plt.plot(x, y)
# plt.xlabel("n_estimator")
# plt.ylabel("concordance index")
# plt.grid(True)

In [7]:
# max_y = max(y)
# max_x = x[y.index(max_y)]
# n_estimators = max_x
# print(f"Max C-index: {max_y:.5f} at n_estimators={max_x}")

## Train & Test

In [8]:
est_cph_tree = GradientBoostingSurvivalAnalysis(
    n_estimators=n_estimators,
    learning_rate=1.0,
    max_depth=1,
    random_state=0
)
est_cph_tree.fit(X_train, y_train)

NameError: name 'n_estimators' is not defined

In [ ]:
c_index_train = est_cph_tree.score(X_train, y_train)
c_index_test_one = est_cph_tree.score(X_test_one, y_test_one)
c_index_test_two = est_cph_tree.score(X_test_two, y_test_two)
c_index_test_three = est_cph_tree.score(X_test_three, y_test_three)

print(f"Train C-index: {c_index_train:.5f}")
print(f"Test 1 C-index: {c_index_test_one:.5f}")
print(f"Test 2 C-index: {c_index_test_two:.5f}")
print(f"Test 3 C-index: {c_index_test_three:.5f}")

## Save Model

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(est_cph_tree, "../models/est_cph_tree.joblib")